<a href="https://colab.research.google.com/github/redinbluesky/handson-llm/blob/main/10_텍스트_임베딩_모델_만들기.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  목차
* [Chapter 10 서론](#chapter10)
* [Chapter 10-1 임베딩 모델](#chapter10-1)
* [Chapter 10-2 대조 학습이란](#chapter10-2)
* [Chapter 10-3 SBERT](#chapter10-3)
* [Chapter 10-4 임베딩 모델 만들기](#chapter10-4)
* [Chapter 10-4-1 대조 샘플 생성하기](#chapter10-4-1)
* [Chapter 10-4-2 모델 훈련](#chapter10-4-2)
* [Chapter 10-4-3 심층 평가](#chapter10-4-3)
* [Chapter 10-4-4 손실함수](#chapter10-4-4)

## Chapter 10 서론   <a class="anchor" id="chapter10"></a>
1. 표현력을 높이고 의미를 잘 포착할 수 있는 임베딩 모델을 만드고 미세 튜닝 방법을 학습한다.


## Chapter 10-1 임베딩 모델 <a class="anchor" id="chapter10-1"></a>
1. 텍스트 데이터를 처리하기 위해서 수치 표현인 벡터로 변경하는데, 이과정을 임베딩이라고 부른다.
    - 이렇게 출력된 벡터도 종종 임베딩이라고 부른다.

2. 임베딩 과정은 임베딩 모델이라고 부르는 LLM을 통해서 수행되며 수행하는 목적에 따라 임베딩을 훈력할 수 있다.
    - 의미론적 유사도를 잘 포착할 수 있도록 훈련할 수 있다.
    - 감성 분류를 위해 텍스트 감성에 맞게 훈련 할 수 있다.


## Chapter 10-2 대조 학습이란 <a class="anchor" id="chapter10-2"></a>
1. 대조 학습은 벡터 공간에서 비슷한 문서가 가까이 노이고 비슷하지 않는 문서는 멀리 떨어지도록 학습하는 방법이다.
    - 비슷한 샘플 쌍과 비슷하지 않은 샘플 쌍을 모델에게 제공한다.

2. 모델이 학습할 때 질문을 이해하는 방식이 중요한데, 단순하게 "왜 P인가요?"가 아니라 "왜 Q가 아니고 P인가요?"라는 질문을 통해서 모델이 더 깊게 이해하도록 유도할 수 있다.
    - 단순한 답을 찾는 것이 아니라 두 개념을 대조시킴으로써 이 개념을 정의하는 특성과 관련 없는 특성을 구분하도록 학습할 수 있다.

3. 무엇이 서로를 구분하게 만드는지와 개념을 구성하는 고유한 특징을 학습하도록 유도할 수 있다.
    - 예를 들어, "왜 고양이는 개가 아니고 고양이인가요?"라는 질문을 통해서 모델이 고양이와 개의 차이를 학습하도록 유도할 수 있다.

4. 대조 학습을 적용하여 텍스트 임베딩모델을 만드는 가장 일반적인 프레임워크는 sentence-transformers이다.

## Chapter 10-3 SBERT <a class="anchor" id="chapter10-3"></a>
1. sentence-transformers 이전에는 BERT와 크로스 인코더 구조를 사용해 문장 임베딩을 만들었다.
    - 두 문장을 동시에 트랜스포머 신경망에 전달하고 두 문장이 얼마나 비슷한지 예측한다.
    - 크로스 인코더는 임베딩을 생성하지 않고 입력 문장 사이의 유사도 점수를 출력한다.
    - 두 문장이 얼마나 유사한지 예측하기 위해 많은 추론 계산이 필요한 오버헤드가 발생한다.

2. sentence-transformers는 분류 헤드를 사용하지 않고 최종 출력 층에 평균 풀링을 사용해 임베딩을 생성한다.
    - 고정 크기의 벡터를 생성하고, 이 벡터를 사용하여 문장 간의 유사도를 계산할 수 있다.

3. sentence-transformers는 가중치를 공유하는 BERT 모델 두 개를 사용하여 두 문장을 동시에 인코딩한다.
    - 두 문장을 동시에 인코딩하고, 임베딩을 생성한 후, 코사인 유사도를 계산하여 두 문장이 얼마나 비슷한지 예측한다.
    - 가중치를 공유하기 때문에 하나의 모델을 사용해서 문장을 차례로 전달할 수 있다.

        ![텍스트_토큰화](./image/10_bi-encoder.png)


4. 문장 쌍의 최적화 과정은 손실 함수를 통해 수행된다.
    - 훈련 과정에서 두 문장의 임베딩과 임베딩의 차이를 연결한 후 소프트맥스 분류기를 사용해 최적화한다.
    - 이 구조를 바이 인코더 또는 SBERT 라고 부른다.

## Chapter 10-4 임베딩 모델 만들기 <a class="anchor" id="chapter10-4"></a>
### Chapter 10-4-1 대조 샘플 생성하기 <a class="anchor" id="chapter10-4-1"></a>
1. 임베딩모델을 사전 훈련할 대 자연어 추론(Natural Language Inference, NLI) 데이터셋을 사용한다.
    - NLI 데이터셋은 문장 쌍과 그 관계를 나타내는 레이블(예: entailment, contradiction, neutral)을 포함한다.
    - 이 데이터셋을 사용하여 모델이 문장 간의 의미적 관계를 학습하도록 한다.
    - 예) 
        - "He is watching Frozen at home"과 "He is watcing in the cinema Coco"라는 문장 쌍은 서로 모순되는 관계(contradiction)를 가지고 있다.
        - "He is watching in the cinema Coco"와 "In the movie theater he is watching the Desney movie Coco"라는 문장은 서로 의미적으로 일치하는 관계(entailment)를 가지고 있다.

In [1]:
# GLUE 벤치마크를 사용한다.
from datasets import load_dataset

# GLUE에서 MLI 데이터 셋을 로드한다.
# 0= entailment(수반), 1=neutral(중립), 2=contradiction(모순)
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50000))
train_dataset = train_dataset.remove_columns(["idx"])

train_dataset[2]

README.md: 0.00B [00:00, ?B/s]

mnli/train-00000-of-00001.parquet:   0%|          | 0.00/52.2M [00:00<?, ?B/s]

mnli/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/1.21M [00:00<?, ?B/s]

mnli/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/1.25M [00:00<?, ?B/s]

mnli/test_matched-00000-of-00001.parquet:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

...

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

Generating test_matched split:   0%|          | 0/9796 [00:00<?, ? examples/s]

Generating test_mismatched split:   0%|          | 0/9847 [00:00<?, ? examples/s]

{'premise': 'One of our number will carry out your instructions minutely.',
 'hypothesis': 'A member of my team will execute your orders with immense precision.',
 'label': 0}

### Chapter 10-4-2 모델 훈련 <a class="anchor" id="chapter10-4-2"></a>
1. 일반적으로 sentence-transformers를 사용하면 되지만, 밑바닥부터 만들기 사전훈련된 훈련된 BERT 모델로 시작한다. 

In [2]:
from sentence_transformers import SentenceTransformer

# BERT 베이스 모델을 사용한다.
embedding_model = SentenceTransformer('bert-base-uncased')

No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.


In [3]:
# 소프트맥스 손실 함수를 정의한다.
from sentence_transformers import losses

train_loss = losses.SoftmaxLoss(
    model=embedding_model,
    sentence_embedding_dimension=embedding_model.get_sentence_embedding_dimension(),
    num_labels=3 # 3개의 레이블(0=entailment, 1=neutral, 2=contradiction)
)

In [7]:
# 모델을 평가하기 위해 STSB(Semantic Textual Similarity Benchmark) 데이터셋을 사용한다.
#   - STSB 데이터셋은 문장 쌍과 그 유사도를 나타내는 점수(0~5)를 포함하고 5에 가까워질 수록 두 문장이 의미적으로 유사함을 나타낸다.
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

val_sts = load_dataset("glue", "stsb", split="validation")
evaluator = EmbeddingSimilarityEvaluator(
    sentences1=val_sts["sentence1"],
    sentences2=val_sts["sentence2"],
    scores=[score/5 for score in val_sts["label"]], # 0~5 점수를 0~1로 정규화
    main_similarity="cosine",
    similarity_fn_names=["cosine", "euclidean", "manhattan", "dot"] # 유사도 계산 방법을 다양하게 사용하여 평가
)

In [10]:
# 훈련 매개변수를 정의한다.
from sentence_transformers import SentenceTransformerTrainingArguments

args = SentenceTransformerTrainingArguments(
    output_dir="./output/embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True, # mixed precision training을 사용하여 훈련 속도를 높인다.
    eval_steps=100,
    logging_steps=100,
    report_to=[] # wandb, tensorboard 등 외부 로깅을 사용하지 않음
)

In [11]:
from sentence_transformers.trainer import SentenceTransformerTrainer

# 임베딩 모델을 훈련한다.
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    evaluator=evaluator,
    loss=train_loss
)

trainer.train()

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Column 'hypothesis' is at index 1, whereas a column with this name is usually expected at index 0. Note that the column order can be important for some losses, e.g. MultipleNegativesRankingLoss will always consider the first column as the anchor and the second as the positive, regardless of the dataset column names. Consider renaming the columns to match the expected order, e.g.:
dataset = dataset.select_columns(['hypothesis', 'entailment', 'contradiction'])


Step,Training Loss
100,1.073100
200,0.938000
300,0.882600
400,0.827500
500,0.814400
600,0.827800
700,0.798500
800,0.786100
900,0.774900
1000,0.760400


TrainOutput(global_step=1563, training_loss=0.8071895606687111, metrics={'train_runtime': 139.7071, 'train_samples_per_second': 357.892, 'train_steps_per_second': 11.188, 'total_flos': 0.0, 'train_loss': 0.8071895606687111, 'epoch': 1.0})

In [ ]:
# 훈련된 모델 평가
#   - pearson_cosine: 코사인 유사도를 사용하여 두 문장 쌍의 의미적 유사성을 평가한 피어슨 상관계수 0.57 정도면 기본 모델로 사용가능하다.
evaluator(embedding_model)

{'pearson_cosine': 0.5700878132425655,
 'spearman_cosine': 0.6414216650718657,
 'pearson_euclidean': 0.6243690223532044,
 'spearman_euclidean': 0.6473540962121631,
 'pearson_manhattan': 0.6336531858330348,
 'spearman_manhattan': 0.65162067388967,
 'pearson_dot': 0.5476969381394715,
 'spearman_dot': 0.5835566340353874,
 'pearson_max': 0.6336531858330348,
 'spearman_max': 0.65162067388967}

### Chapter 10-4-3 심층 평가 <a class="anchor" id="chapter10-4-3"></a>
1. 임베딩 모델을 평가하기 위해서 MTEB(Multi-Task Benchmark) 벤치마크를 사용한다.
    - MTEB는 56개의 다운스트림 작업을 포함하고 있으며, 112개의 데이터셋과 5개의 언어를 포함고 8개의 임베딩 평가 작업으로 구성된다.
    - MTEB 리더보드에서 다양한 임베딩 모델의 성능을 비교할 수 있다.

In [15]:
import mteb
from pprint import pprint

# 평가 작업을 선택한다.
tasks = mteb.get_tasks(tasks=["Banking77Classification.v2"])

# 결과를 계산한다.
results = mteb.evaluate(embedding_model, tasks=tasks)
for task_result in results:
    pprint(task_result.only_main_score().to_dict())

Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/294k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/91.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9993 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3076 [00:00<?, ? examples/s]

{'dataset_revision': '18072d2685ea682290f7b8924d94c62acc19c0b2',
 'date': datetime.datetime(2026, 7, 13, 4, 49, 54, 767236, tzinfo=datetime.timezone.utc),
 'evaluation_phases': [{'end': 5.440599165000094,
                        'name': 'Data loading',
                        'split': '',
                        'start': 7.97500069893431e-06,
                        'subset': ''},
                       {'end': 5.440615708999758,
                        'name': 'Dataset transform',
                        'split': '',
                        'start': 5.440614482000456,
                        'subset': ''},
                       {'end': 8.397846170000776,
                        'name': 'Encoding training samples',
                        'split': 'test',
                        'start': 6.520133205000093,
                        'subset': 'default'},
                       {'end': 9.708443614999851,
                        'name': 'Encoding test samples',
                        'spl

### Chapter 10-4-4 손실함수 <a class="anchor" id="chapter10-4-4"></a>
1. 코사인 유사도 손실
    - 일반적으로 텍스트의 의미론적 유사도 작업에 사용된다.
    - 두 문장의 임베딩 벡터 사이의 코사인 유사도를 계산하고, 이 유사도를 기반으로 손실을 계산한다.
    - 0과 1로 완전하게 구분하지 않으며 어느정도 비슷한지를 평가할 수 있다.
    - 두 텍스트 임베딩 사이의 코사인 유사도를 계산하고 레이블로 제공된 유사도 점수와 비교한다.

2. NLI 데이터 셋에서 코사인 유사도를 사용하려먼 
    - entailment(수반) 레이블은 1, neutral(중립) 레이블은 0, contradiction(모순) 레이블은 0.0으로 매핑한다.
    - 모델이 두 문장의 의미적 유사성을 학습하도록 한다.

In [18]:
# GLUE 벤치마크를 사용한다.
from datasets import load_dataset, Dataset

# GLUE에서 MLI 데이터 셋을 로드한다.
# 0= entailment(수반), 1=neutral(중립), 2=contradiction(모순)
train_dataset = load_dataset("glue", "mnli", split="train").select(range(50000))
train_dataset = train_dataset.remove_columns(["idx"])

# 중립/모순=0, 수반=1로 매핑한다.
mapping = {0: 1, 1: 0, 2: 0}
train_dataset = Dataset.from_dict({
    "sentence1": train_dataset["premise"],
    "sentence2": train_dataset["hypothesis"],
    "label": [float(mapping[label]) for label in train_dataset["label"]]
})

In [21]:
embedding_model = SentenceTransformer('bert-base-uncased')

train_loss = losses.CosineSimilarityLoss(
    model=embedding_model
)

args = SentenceTransformerTrainingArguments(
    output_dir="./outputCosine/embedding_model",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    fp16=True, # mixed precision training을 사용하여 훈련 속도를 높인다.
    eval_steps=100,
    logging_steps=100,
    report_to=[] # wandb, tensorboard 등 외부 로깅을 사용하지 않음
)

# 임베딩 모델을 훈련한다.
trainer = SentenceTransformerTrainer(
    model=embedding_model,
    args=args,
    train_dataset=train_dataset,
    evaluator=evaluator,
    loss=train_loss
)

trainer.train()



No sentence-transformers model found with name bert-base-uncased. Creating a new one with mean pooling.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
100,0.231700
200,0.170800
300,0.171700
400,0.156300
500,0.155000
600,0.158200
700,0.151500
800,0.155700
900,0.151000
1000,0.148100


TrainOutput(global_step=1563, training_loss=0.15752250112483537, metrics={'train_runtime': 138.668, 'train_samples_per_second': 360.573, 'train_steps_per_second': 11.272, 'total_flos': 0.0, 'train_loss': 0.15752250112483537, 'epoch': 1.0})

In [22]:
evaluator(embedding_model)

{'pearson_cosine': 0.7268905387392256,
 'spearman_cosine': 0.7292076954048319,
 'pearson_euclidean': 0.731506660787363,
 'spearman_euclidean': 0.7322831121179572,
 'pearson_manhattan': 0.732586674279251,
 'spearman_manhattan': 0.7333696876009106,
 'pearson_dot': 0.6867554757781075,
 'spearman_dot': 0.6884100337979729,
 'pearson_max': 0.732586674279251,
 'spearman_max': 0.7333696876009106}

In [23]:
# 결과를 계산한다.
results = mteb.evaluate(embedding_model, tasks=tasks)
for task_result in results:
    pprint(task_result.only_main_score().to_dict())

Evaluating tasks:   0%|          | 0/1 [00:00<?, ?it/s]

{'dataset_revision': '18072d2685ea682290f7b8924d94c62acc19c0b2',
 'date': datetime.datetime(2026, 7, 13, 4, 49, 54, 767236, tzinfo=TzInfo(0)),
 'evaluation_phases': [{'end': 5.440599165000094,
                        'name': 'Data loading',
                        'split': '',
                        'start': 7.97500069893431e-06,
                        'subset': ''},
                       {'end': 5.440615708999758,
                        'name': 'Dataset transform',
                        'split': '',
                        'start': 5.440614482000456,
                        'subset': ''},
                       {'end': 8.397846170000776,
                        'name': 'Encoding training samples',
                        'split': 'test',
                        'start': 6.520133205000093,
                        'subset': 'default'},
                       {'end': 9.708443614999851,
                        'name': 'Encoding test samples',
                        'split': 'test',